### Statistical Tests for Experiment 1

In [40]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import pandas as pd
import numpy as np

In [29]:
#Prepare data

def prepare_data(csv_path):
    df = pd.read_csv(csv_path)
    df = df.replace([float('inf'), float('-inf')], float('nan')).dropna()
    # Extract only the numeric value from "Mean±Std" columns
    # Example: '0.549±0.477' -> 0.549
    for col in df.columns:
        if df[col].dtype == 'object' and '±' in str(df[col].iloc[0]):
            df[col] = df[col].apply(lambda x: float(str(x).split('±')[0]))
    return df

In [44]:
# Data Preprocessing

df = prepare_data('../dataset/sets/exp1_sets.csv')

# 2. Re-check for any remaining NaNs or Infs explicitly
df = df.dropna(subset=['Execution_Time', 'Algorithm', 'Grid_Size'])
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['Execution_Time'])

display(df)

,Experiment,Algorithm,Grid_Size,Density,Execution_Time,Nodes_Expanded,Path_Length,Memory_B
0,1,BFS,Small(10x10),Small(10%),0.220,86,11.0,4128
1,1,BFS,Small(10x10),Small(10%),0.415,92,11.0,4416
2,1,BFS,Small(10x10),Small(10%),1.065,86,11.0,4272
3,1,BFS,Small(10x10),Small(10%),0.400,80,10.0,4272
4,1,BFS,Small(10x10),Small(10%),0.395,91,11.0,4416
...,...,...,...,...,...,...,...,...
355,1,D*_Lite,Large(50x50),Large(20%),16.920,725,57.0,41040
356,1,D*_Lite,Large(50x50),Large(20%),9.620,400,55.0,25872
357,1,D*_Lite,Large(50x50),Large(20%),11.610,450,54.0,28032
358,1,D*_Lite,Large(50x50),Large(20%),6.035,329,54.0,24096


In [54]:
# Two Way Anova

# DV: Execution_Time | Factors: Algorithm, Grid_Size
model = ols('Execution_Time ~ C(Algorithm) * C(Grid_Size)', data=df).fit()
anova_results = sm.stats.anova_lm(model, typ=2)
print("Two way Anova Analysis:\n")
display(anova_results)

Two way Anova Analysis:



,sum_sq,df,F,PR(>F)
C(Algorithm),874.996129,3.0,72.352678,2.533431e-36
C(Grid_Size),7141.963516,2.0,885.844234,1.253058e-136
C(Algorithm):C(Grid_Size),803.203381,6.0,33.208098,1.461119e-31
Residual,1390.750946,345.0,NaN,NaN


In [52]:
# Tukey HSD
tukey = pairwise_tukeyhsd(endog=df['Execution_Time'], groups=df['Algorithm'], alpha=0.05)
print("\nTukey HSD Results:\n", tukey)


Tukey HSD Results:
  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
 group1  group2  meandiff p-adj   lower  upper  reject
------------------------------------------------------
     A*      BFS   3.5669    0.0  1.5769 5.5569   True
     A*  D*_Lite   3.0496 0.0005  1.0596 5.0395   True
     A* Dijkstra   4.1298    0.0  2.1454 6.1143   True
    BFS  D*_Lite  -0.5174  0.908 -2.5074 1.4726  False
    BFS Dijkstra   0.5629 0.8841 -1.4215 2.5474  False
D*_Lite Dijkstra   1.0803 0.4969 -0.9042 3.0647  False
------------------------------------------------------
